# Q-Learning — Mountain Car Continuous (LOST)

**Metodología iterativa train/test.** En vez de entrenar una configuración entera y testear una sola vez al final, intercalamos: entrenamos un bloque de episodios, evaluamos (greedy), registramos estadísticas, y repetimos. Así obtenemos la **curva de aprendizaje** del agente y su variabilidad.

Las configuraciones están **agrupadas por decisión** (un factor variado por vez alrededor de la ganadora), de modo que la comparativa de cada grupo justifica una decisión por sí sola.

In [ ]:
import gymnasium as gym
import numpy as np
import matplotlib.pyplot as plt

from q_learning_agent import QLearningAgent
from configs import build_configs, configs_in_group, make_get_state, GROUPS
from train_eval_loop import run_train_eval, resume_train_eval
from make_plots import make_group_plots
import plotting

ENV_ID = 'MountainCarContinuous-v0'

## 1. Entornos y configuraciones (agrupadas por decisión)

In [ ]:
train_env = gym.make(ENV_ID, render_mode='rgb_array')
eval_env  = gym.make(ENV_ID, render_mode='rgb_array')

configs = build_configs()
print(f'{len(configs)} configuraciones en {len(GROUPS)} grupos:\n')
for g, title in GROUPS.items():
    nombres = [c['name'] for c in configs_in_group(g, configs)]
    print(f"  {g:16s} ({title}):\n     {nombres}")

## 2. Loop train/test para UNA configuración

`run_train_eval` corre `n_cycles` ciclos de `train_episodes_per_cycle` + `eval_episodes` de test.
Para reproducir el barrido original (10 000 episodios): `n_cycles=10, train=1000, eval=100`.
**Aviso:** varios minutos por config. Para prueba rápida bajá los números (`n_cycles=5, train=200, eval=20`).

In [ ]:
cfg = configs[0]  # base (ganadora)
get_state = make_get_state(cfg['x_space'], cfg['vel_space'])
actions = cfg['actions']

agent = QLearningAgent(len(cfg['x_space'])+1, len(cfg['vel_space'])+1, len(actions),
                       seed=0, **cfg['hyper'])
result = run_train_eval(agent, train_env, eval_env, get_state, actions,
                        n_cycles=10, train_episodes_per_cycle=1000, eval_episodes=100,
                        config_name=cfg['name'], checkpoint_dir='checkpoints', eval_seed=0)
plotting.plot_full_report(result, band='std', outdir='checkpoints/figs')

## 3. Correr TODAS las configuraciones

Corre las 11 configs y guarda checkpoints + historiales. Es lo más pesado (varios minutos por config). Podés correr un grupo por vez con `configs_in_group('acciones')` en distintas sesiones.

In [ ]:
# Para un grupo puntual: configs_to_run = configs_in_group('acciones')
configs_to_run = build_configs()  # todas

for i, c in enumerate(configs_to_run, 1):
    print(f"== {i}/{len(configs_to_run)}: {c['name']}")
    a = QLearningAgent(len(c['x_space'])+1, len(c['vel_space'])+1, len(c['actions']),
                       seed=0, **c['hyper'])
    gs = make_get_state(c['x_space'], c['vel_space'])
    run_train_eval(a, train_env, eval_env, gs, c['actions'],
                   n_cycles=10, train_episodes_per_cycle=1000, eval_episodes=100,
                   config_name=c['name'], checkpoint_dir='checkpoints', eval_seed=0)

## 4. Comparativas por eje de decisión

Una figura por grupo (`checkpoints/figs/grupo_<grupo>.png`). Cada una justifica una decisión: bins de posición, discretización de velocidad, número de acciones, gamma, alpha y exploración.

In [ ]:
# Genera checkpoints/figs/grupo_<grupo>.png (una figura por eje de decisión)
make_group_plots('checkpoints', band='std')

# Listar las comparativas generadas
import os
for f in sorted(os.listdir('checkpoints/figs')):
    if f.startswith('grupo_'):
        print(f)

# Para ver una comparativa concreta dentro del notebook, por ejemplo 'acciones':
items = [(c['name'], None) for c in configs_in_group('acciones')]
# (las imágenes quedan guardadas en checkpoints/figs/)

## 5. Reanudar desde un checkpoint

Cada `.pkl` guarda la tabla Q, epsilon y episodios entrenados (sirve como *modelo computado* de la entrega).

In [ ]:
ckpt = result['checkpoint_path']
result2, agent2 = resume_train_eval(ckpt, train_env, eval_env, get_state, actions,
                                    n_cycles=5, train_episodes_per_cycle=1000, eval_episodes=100,
                                    checkpoint_dir='checkpoints', eval_seed=0)
plotting.plot_learning_curve(result2['history'], band='std', label=cfg['name']+' (reanudado)')
plt.show()

## 6. Ver la política aprendida (modo humano)

In [ ]:
human_env = gym.make(ENV_ID, render_mode='human')
vis = agent.evaluate(human_env, get_state, actions, episodes=5, seed=0)
print('Recompensas:', [round(r,2) for r in vis['rewards']])
print('Éxitos:', vis['successes'])
human_env.close()